In [1]:
import pandas as pd
import sys
from pathlib import Path

sys.path.append(str(Path("..").resolve()))

DEMENTIA_PATH = Path("..") / "data" / "dementia_data.csv"
TRANSCRIPTS_PATH = Path("..") / "data" / "transcripts_cleaned.csv"

# Build final CTD dataframe for logistic regression model

# original dataset with precomputed Transcript_CTD features
dementia_df = pd.read_csv(DEMENTIA_PATH)

ctd_features = dementia_df.dropna(subset=["Transcript_CTD"]).copy()

# Make found_fillers numeric 0/1
# overwrite the original list-valued column
ctd_features["found_fillers"] = (ctd_features["filler_count"] > 0).astype(int)

# Make repetitions numeric using token_count - type_count
# overwrite the original dict-valued column
ctd_features["repetitions"] = ctd_features["token_count"] - ctd_features["type_count"]

# Preprocessed/cleaned transcripts
clean_df = pd.read_csv(TRANSCRIPTS_PATH)
ctd_clean_df = clean_df.dropna(subset=["Transcript_CTD"]).copy()

# Merge cleaned transcripts onto CTD features via Record-ID
ctd_merged = ctd_clean_df.merge(
    ctd_features[
        [
            "Record-ID",
            "Gender",
            "Age",
            "Converted-MMSE",
            "filler_count",
            "found_fillers",
            "token_count",
            "type_count",
            "type_token_ratio",
            "ma_ttr",
            "brunets_index",
            "content_density",
            "repetitions",
            "sentence_count",
            "average_words_per_sentence",
        ]
    ],
    on="Record-ID",
    how="left",
)

# Final columns for the CTD LR model
ctd_keep_cols = [
    "Record-ID",
    "Class",      
    "Label",          
    "Gender",
    "Age",
    "Converted-MMSE",
    "Transcript_CTD",  
    "filler_count",
    "found_fillers",   
    "token_count",
    "type_count",
    "type_token_ratio",
    "ma_ttr",
    "brunets_index",
    "content_density",
    "repetitions",      
    "sentence_count",
    "average_words_per_sentence",
]

# Intersect with existing columns for safety
ctd_keep_cols = [c for c in ctd_keep_cols if c in ctd_merged.columns]

ctd_df = ctd_merged[ctd_keep_cols].copy()

print("CTD final shape:", ctd_df.shape)
ctd_df.head()


CTD final shape: (156, 18)


,Record-ID,Class,Label,Gender,Age,Converted-MMSE,Transcript_CTD,filler_count,found_fillers,token_count,type_count,type_token_ratio,ma_ttr,brunets_index,content_density,repetitions,sentence_count,average_words_per_sentence
0,Process-rec-002,MCI,1,1,61,25.0,<pause_medium> there’s a lad stood on the stoo...,2,1,76,46,0.605263,1.000000,1.487174,0.421053,30,5,15.200000
1,Process-rec-003,MCI,1,0,62,29.0,"<pause_medium> um, the picture is of a kitchen...",6,1,150,84,0.560000,0.995495,1.620714,0.433333,66,7,21.428571
2,Process-rec-004,MCI,1,0,67,29.0,"a mother presumably, or a fe, an adult female ...",4,1,162,88,0.543210,1.000000,1.675909,0.425926,74,4,40.500000
3,Process-rec-005,MCI,1,1,65,27.0,"‘50s style er scene of domestic um confusion, ...",4,1,44,36,0.818182,1.000000,1.057222,0.500000,8,1,44.000000
4,Process-rec-006,Dementia,2,1,83,26.0,"<pause_short> fa, a family <pause_short> in th...",3,1,122,78,0.639344,0.994444,1.399103,0.401639,44,5,24.400000


In [6]:
from utils import compute_linguistic_features_for_column

# Replicate Transcript_CTD's additional features from original dataset for Transcript_PFT and Transcript_SFT

# Keep just the metadata columns from the original dataset
meta_cols = ["Record-ID", "Gender", "Age", "Converted-MMSE"]
meta_df = dementia_df[meta_cols].copy()

# Merge metadata into cleaned transcripts on Record-ID
base_df = clean_df.merge(meta_df, on="Record-ID", how="left")

features_by_transcript = {}

for col in ["Transcript_PFT", "Transcript_SFT"]:
    # Drop rows where this transcript type is missing
    df_col = base_df.dropna(subset=[col]).copy()

    df_with_features = compute_linguistic_features_for_column(df_col, col)

    # Final columns for the PFT/SFT LR model
    keep_cols = [
        "Record-ID",
        "Class",   
        "Label",      
        "Gender",
        "Age",
        "Converted-MMSE",
        col,           
        "token_count",
        "type_count",
        "type_token_ratio",
        "ma_ttr",
        "brunets_index",
        "sentence_count",
        "average_words_per_sentence",
        "filler_count",
        "found_fillers",
        "content_density",
        "repetitions",
    ]

    keep_cols = [c for c in keep_cols if c in df_with_features.columns]

    df_final = df_with_features[keep_cols].copy()

    features_by_transcript[col] = df_final

pft_df = features_by_transcript["Transcript_PFT"]
sft_df = features_by_transcript["Transcript_SFT"]

print("PFT shape:", pft_df.shape)
print("SFT shape:", sft_df.shape)
# pft_df.head()
sft_df.head()


PFT shape: (157, 18)
SFT shape: (152, 18)


,Record-ID,Class,Label,Gender,Age,Converted-MMSE,Transcript_SFT,token_count,type_count,type_token_ratio,ma_ttr,brunets_index,sentence_count,average_words_per_sentence,filler_count,found_fillers,content_density,repetitions
0,Process-rec-001,MCI,1,1,62,25.0,"<pause_medium> giraffe, kangaroo, lion, tiger,...",30,21,0.700000,0.700000,7.831046,2,15.0,0,0,0.666667,9
1,Process-rec-002,MCI,1,1,61,25.0,"<pause_short> dogs, cats, birds <pause_short> ...",36,26,0.722222,0.722222,8.111938,2,18.0,1,1,0.638889,10
2,Process-rec-003,MCI,1,0,62,29.0,"cow, bull, ewe, ram, chicken, goose, um <sigh>...",48,39,0.812500,0.812500,8.289977,2,24.0,5,1,0.729167,9
3,Process-rec-004,MCI,1,0,67,29.0,um <pause_short> impala <pause_short> er cheet...,48,30,0.625000,0.625000,9.103132,2,24.0,10,1,0.500000,18
4,Process-rec-005,MCI,1,1,65,27.0,"dog, cat, giraffe, wallaby, kangaroo, tortoise...",62,40,0.645161,0.735000,9.444925,1,62.0,15,1,0.612903,22
